In [ ]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset
import matplotlib.pyplot as plt
import os
from utility.ffqd_mnist import FFQD_Dataset, label_for_id
from torch.utils.data import DataLoader
import torchvision.transforms as T

def image_and_pad (x):
    return nn.functional.pad(torch.Tensor(x).view(1,28,28),(2,2,2,2))

DATA_PATH='/leonardo_work/tra26_ictpai/hen_forma_data/ffqd_mnist/'
if not os.path.exists(DATA_PATH):
    DATA_PATH='ffqd_mnist/'

training_data = FFQD_Dataset(DATA_PATH, train=True, transform=image_and_pad)
train_dataloader = DataLoader(training_data, batch_size=64)

test_data = FFQD_Dataset(DATA_PATH, transform=image_and_pad)
test_dataloader = DataLoader(test_data, batch_size=64)
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

In [ ]:
training_data[1][0].shape

In [ ]:
# by: https://blog.paperspace.com/writing-lenet5-from-scratch-in-python/
class LeNet(torch.nn.Module):
    def __init__(self, output_size=10):
        super(LeNet, self).__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(1, 6, kernel_size=5, stride=1, padding=0),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size = 2, stride = 2)
        )
        # 14 x 14 x 6
        self.layer2 = nn.Sequential(
            nn.Conv2d(6, 16, kernel_size=5, stride=1, padding=0),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size = 2, stride = 2)
        )
        # 5 x 5 x 16
        # see: https://madebyollin.github.io/convnet-calculator/

        self.fc1 = nn.Linear(400, 120)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(120, 84)
        self.relu1 = nn.ReLU()
        self.fc3 = nn.Linear(84, output_size)


    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = out.reshape(out.size(0), -1)
        out = self.fc1(out)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.relu1(out)
        out = self.fc3(out)
        return out
        # return y

In [ ]:
model = LeNet().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=.01)
criterion = torch.nn.CrossEntropyLoss()

def reset_model():
    global model, optimizer
    model = LeNet().to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=.01)


In [ ]:
def train_one_epoch(dataloader, model, loss_fn, optimizer):
    model.train()
    for batch_id, (X,y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # print(X.shape)
        # calculate loss
        y_pred = model(X)
        loss = loss_fn(y_pred, y)


        # print( y_pred.dtype, y_pred.shape)
        # print(loss)

        # backprop loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if batch_id % 100 == 0:
            print(f"({batch_id}) loss: {loss:>7}")


In [ ]:
def test_one_epoch(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X,y in dataloader:
            X, y = X.to(device), y.to(device)
            pred_y = model(X)
            test_loss += loss_fn(pred_y, y).item()
            correct += (pred_y.argmax(1) == y.argmax(1)).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    return test_loss, correct


In [ ]:
reset_model()
# old_weights = torch.Tensor(np.array(model.get_parameter("fc.weight").detach().numpy())) # copy old weights
# old_weights = model.get_parameter("mlp_stack.0.weight").clone().detach()
for i in range(10):
    train_one_epoch(train_dataloader, model, criterion, optimizer)
    print(test_one_epoch(test_dataloader, model, criterion))

In [ ]:
for name, par in model.named_parameters():
    print(name, par.shape)

In [ ]:
l0_weights = model.get_parameter('layer2.0.weight').detach().cpu()
# l0_weights_diff = l0_weights- old_weights
f, axs = plt.subplots(10,10,figsize=(12,12))
axs = axs.flatten()
for i in range(6):
    for ii in range(1):
        axs[i*6+ii].imshow(l0_weights[i,ii], cmap="gray")
        axs[i*6+ii].axis('off')

In [ ]:
test_batch, test_batch_label = next(iter(test_dataloader))

image_id = 44

image = test_batch[image_id].unsqueeze(0)
label = test_batch_label[image_id].unsqueeze(0)

fig, axs = plt.subplots(1,2, figsize=(6,6))
axs= axs.flatten()

axs[0].imshow(image.view(32,32), cmap='gray')
model.eval()
x_in = image.clone().to(device).requires_grad_()
print(x_in.shape)
model.zero_grad()
pred = model(x_in)
# loss = criterion(pred, label)
# loss.backward()

pred_index = pred.argmax()

pred_max = pred[0, pred_index]
# print(pred_max)
pred_max.backward()

# # # saliency = nn.ReLU()(x_in.grad)
saliency = x_in.grad.detach().abs()[0].cpu()
# # # print(saliency.shape, saliency)
saliency = saliency.reshape(32, 32)
# #
axs[1].imshow(saliency, cmap="hot")


plt.show()

In [ ]:
test_batch, test_batch_label = next(iter(test_dataloader))

item_id = 23 # 23, 55(rob), 34(rob)

image = test_batch[item_id].unsqueeze(0)
print(f"ground truth: {np.argmax(test_batch_label[item_id])} / {label_for_id(np.argmax(test_batch_label[item_id]))}")

fig, axs = plt.subplots(2,2, figsize=(6,6))
axs= axs.flatten()

axs[0].imshow(image.view(32,32), cmap='gray')

x_in = image.clone().to(device).requires_grad_()
model.zero_grad()
pred = model(x_in)
pred_index = pred.argmax()
print(f"predcition: {pred_index.item()} / {label_for_id(pred_index.item())}")
pred_max = pred[0, pred_index]
# print(pred_max)
pred_max.backward()


saliency = x_in.grad.detach().abs()[0].cpu()
# print(saliency.shape, saliency)
saliency = saliency.reshape(32, 32)
axs[1].imshow(saliency, cmap="hot")

model.zero_grad()
pred = model(x_in)
pred_attack = pred[0, 0]
pred_attack.backward()
# print(x_in.grad)
x_attack = 0.5 *image.to(device) + 0.7*(x_in.grad.detach()[0]) #nn.ReLU()

axs[2].imshow(x_attack.view(32,32).cpu(), cmap='gray')
pred = model(x_attack)
pred_index = pred.argmax()
print(f'attacked prediction: {pred_index.item()} / {label_for_id(pred_index.item())}')

plt.show()
